In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df = pd.read_csv("../Dataset/DateFruit_Dataset.csv")

In [3]:
df.head() # 34 features, 1 category

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [6]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
le = LabelEncoder()
y = le.fit_transform(y)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## ANN

In [10]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [11]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## Build Model

In [13]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7),
        )

    def forward(self, x):
        return self.model(x)

In [14]:
model = ANN()

# Loss & Optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [15]:
epochs = 388
for epoch in range(epochs):
    model.train()

    runnig_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        output = model(xb)
        loss = criteria(output, yb)
        loss.backward()
        optimizer.step()

        runnig_loss += loss.item()

    train_loss = runnig_loss/len(train_loader)
    print(f"epoch = {epoch}, loss = {train_loss}")

epoch = 0, loss = 1.7025981882344121
epoch = 1, loss = 1.0230667824330537
epoch = 2, loss = 0.6784594136735668
epoch = 3, loss = 0.5205889493227005
epoch = 4, loss = 0.4432186186313629
epoch = 5, loss = 0.41383568877759186
epoch = 6, loss = 0.37054010253885517
epoch = 7, loss = 0.3340663689634074
epoch = 8, loss = 0.3147003015746241
epoch = 9, loss = 0.29174986416878906
epoch = 10, loss = 0.26871691061102826
epoch = 11, loss = 0.258237273796745
epoch = 12, loss = 0.24248595017453897
epoch = 13, loss = 0.2271004811577175
epoch = 14, loss = 0.21536713417457498
epoch = 15, loss = 0.20443736049144165
epoch = 16, loss = 0.19315207101728604
epoch = 17, loss = 0.19019103957259137
epoch = 18, loss = 0.17831755056977272
epoch = 19, loss = 0.1716525535220685
epoch = 20, loss = 0.16293144096498904
epoch = 21, loss = 0.15877344459295273
epoch = 22, loss = 0.15272925615958546
epoch = 23, loss = 0.15172226807993391
epoch = 24, loss = 0.1473739433871663
epoch = 25, loss = 0.14386989062894945
epoch = 

## Evaluate

In [16]:
model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        output = model(xb)
        actual, predicted = torch.max(output, 1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)
print("Total values: ", total)
print("Correct Value: ", correct)
print("Accuracy: ", correct/total * 100)

Total values:  180
Correct Value:  167
Accuracy:  92.77777777777779


In [17]:
y_test.shape

(180,)